# A1 homework: an ML design doc for appointment no-shows

**Scenario.** A community health clinic runs about 1,000 appointments a month. About one patient in six
does not turn up and does not cancel: a *no-show*. Each no-show wastes a slot that the clinic values at
**£120** (staff time, plus a patient on the waiting list who could have been seen). The operations manager
asks: *"Can we use ML to reduce no-shows?"*

What the clinic can do: two days before an appointment, a receptionist can **phone** the patient. A call
takes about 10 minutes of staff time, which the clinic costs at **£8**. In a small pilot, a reminder call
cut the chance of a no-show by about **30%** (relative): a patient with a 20% no-show risk drops to 14%.
Calling everyone is possible but expensive and annoys patients who would have come anyway.

You will act as the ML engineer and produce what this module taught: a framing, metrics, baselines (and
build them), a cost-based threshold, a risk review and a **one-page design doc**, all in this notebook.

The data are **synthetic**, generated in the next cell from a fixed seed, so everyone gets the same numbers.
The clinic did **no** reminder calls during the year the data cover.

**How to work:** fill in each `TODO` cell, then run the check cell below it; it prints `OK` when your code
is right. Markdown cells marked *Your answer* are for your writing. At the end, restart the kernel and run
all cells.

Runs on any laptop or free Colab (CPU) in a few seconds. You need NumPy, pandas, matplotlib and
scikit-learn 1.2 or newer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_appointments(n=12_000, seed=42):
    rng = np.random.default_rng(seed)
    n_patients = 4_000
    patient = rng.integers(0, n_patients, n)
    p_age = rng.integers(18, 90, n_patients)
    p_distance = np.round(rng.gamma(2.0, 4.0, n_patients), 1)
    p_band = rng.integers(1, 6, n_patients)               # area deprivation band, 1 = least deprived
    p_sms = rng.random(n_patients) < 0.7
    p_risk = rng.normal(0, 1.0, n_patients)               # unobserved personal tendency
    appt_day = np.sort(rng.integers(0, 360, n))
    lead = np.clip(rng.geometric(1 / 14, n), 2, 90)
    df = pd.DataFrame({
        "appointment_id": np.arange(n),
        "patient_id": patient,
        "appt_date": pd.Timestamp("2025-01-01") + pd.to_timedelta(appt_day, unit="D"),
        "lead_days": lead,
        "age": p_age[patient],
        "distance_km": p_distance[patient],
        "deprivation_band": p_band[patient],
        "sms_opt_in": p_sms[patient],
        "appt_type": rng.choice(["new", "follow-up", "procedure"], n, p=[0.3, 0.55, 0.15]),
    })
    df["booked_on"] = df["appt_date"] - pd.to_timedelta(df["lead_days"], unit="D")
    df["weekday"] = df["appt_date"].dt.day_name().str[:3]
    young = (df["age"] < 30).astype(float)
    logit = (-2.9 + 0.045 * df["lead_days"] + 0.6 * young - 0.012 * (df["age"] - 50).clip(lower=0)
             + 0.04 * df["distance_km"] + 0.18 * (df["deprivation_band"] - 3) - 0.4 * df["sms_opt_in"]
             + 0.5 * (df["appt_type"] == "new") - 0.6 * (df["appt_type"] == "procedure")
             + 0.3 * df["weekday"].isin(["Mon", "Fri"]) + p_risk[patient])
    df["no_show"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    # Outcomes are typed into the system by admin staff, some days later.
    df["outcome_recorded_on"] = df["appt_date"] + pd.to_timedelta(rng.poisson(2.5, n) + rng.integers(0, 2, n) * rng.integers(0, 12, n), unit="D")
    df["prior_appts"] = 0
    df["prior_no_shows"] = 0
    for _, appointments in df.groupby("patient_id"):
        cutoff = (appointments["appt_date"] - pd.Timedelta(days=2)).to_numpy()
        past = np.sort(appointments["appt_date"].to_numpy())
        missed = np.sort(appointments.loc[appointments["no_show"] == 1, "outcome_recorded_on"].to_numpy())
        # Dates have day precision. Count events strictly before the prediction date.
        df.loc[appointments.index, "prior_appts"] = np.searchsorted(past, cutoff, side="left")
        df.loc[appointments.index, "prior_no_shows"] = np.searchsorted(missed, cutoff, side="left")
    # Recorded at the front desk on the day: only exists for patients who turned up.
    df["minutes_late"] = np.where(df["no_show"] == 1, np.nan, np.round(np.clip(rng.normal(2, 8, n), -15, 60)))
    cols = ["appointment_id", "patient_id", "booked_on", "appt_date", "weekday", "lead_days", "appt_type", "age",
            "distance_km", "deprivation_band", "sms_opt_in", "prior_appts", "prior_no_shows", "minutes_late",
            "outcome_recorded_on", "no_show"]
    return df[cols]


df = make_appointments()
print(df.shape)
df.head()

## The columns

| Column | Meaning |
|---|---|
| `appointment_id`, `patient_id` | identifiers |
| `booked_on`, `appt_date`, `weekday` | when the appointment was booked, when it takes place, and its weekday |
| `lead_days` | days between booking and the appointment |
| `appt_type` | `new`, `follow-up` or `procedure` |
| `age`, `distance_km` | patient age and distance from home to the clinic |
| `deprivation_band` | deprivation of the patient's home area, 1 (least deprived) to 5 (most) |
| `sms_opt_in` | whether the patient agreed to receive text messages |
| `prior_appts`, `prior_no_shows` | appointments before the prediction date and no-shows recorded before that date |
| `minutes_late` | how late the patient checked in at the front desk |
| `outcome_recorded_on` | the date admin staff typed the outcome into the system |
| `no_show` | 1 if the patient did not attend: the label |

## Task 1: frame the problem

**1a. Your answer.** Fill in the framing table (replace each *...*). Keep each answer to one line.

| Item | Your answer |
|---|---|
| Business objective | ... |
| Decision the prediction drives | ... |
| Prediction | ... |
| Unit of prediction | ... |
| Prediction time | ... |
| Label definition | ... |
| Known proxy gap | ... |

**1b. Code.** Compute:

- `no_show_rate`: the share of appointments that were no-shows;
- `delay`: a Series with the label delay in whole days, from `appt_date` to `outcome_recorded_on`;
- `delay_median` and `delay_p95`: its median and 95th percentile;
- `FEATURES`: the list of columns you may use as model inputs. Only columns known **two days before the
  appointment** qualify. Bookings in this dataset are made at least two days ahead. Leave out identifiers, raw dates, the label and anything recorded later.

In [ ]:
# TODO
no_show_rate = None
delay = None
delay_median = None
delay_p95 = None
FEATURES = []

In [ ]:
# Check
assert no_show_rate is not None and abs(no_show_rate - df["no_show"].mean()) < 1e-12, "no_show_rate should be the mean of no_show"
assert delay is not None and len(delay) == len(df) and int(delay.min()) >= 0, "delay should be one non-negative number of days per row"
assert delay_median == 4 and delay_p95 == 13, f"Expected a median of 4 and a p95 of 13 days, got {delay_median} and {delay_p95}"
_bad = {"minutes_late", "outcome_recorded_on", "no_show", "appointment_id", "patient_id", "booked_on", "appt_date"}
assert not set(FEATURES) & _bad, f"These columns must not be features: {sorted(set(FEATURES) & _bad)}"
assert set(FEATURES) <= set(df.columns), f"Unknown columns: {sorted(set(FEATURES) - set(df.columns))}"
assert len(FEATURES) >= 6, "Use at least six of the allowed columns"
print("OK")

Why does `minutes_late` have to go? Run the next cell: on its own, "is `minutes_late` missing?" predicts
no-shows **perfectly**, because the front desk only records it for patients who turn up. It is a textbook
leak: it is only known *after* the event you want to predict.

**1c. Your answer.** `prior_no_shows` is allowed, but there is a subtlety: outcomes are typed in a few days
late (your `delay`). In production, which earlier appointments could the count actually include, two days
before an appointment? Explain how the generator enforces that cutoff and why counting earlier appointments by appointment date alone would leak labels. (2 to 3 sentences.)

*Your answer:* ...

In [ ]:
print("AUC of 'minutes_late is missing':", roc_auc_score(df["no_show"], df["minutes_late"].isna()))

## Task 2: split by time

The model will be used on *future* appointments, so evaluate it that way. Create:

- `train`: appointments before 2025-09-01;
- `val`: appointments from 2025-09-01 up to (not including) 2025-11-01;
- `test`: appointments from 2025-11-01 on. Do not look at test results until Task 5.

In [ ]:
# TODO
train = None
val = None
test = None

In [ ]:
# Check
assert (len(train), len(val), len(test)) == (8086, 2044, 1870), f"Expected 8086 / 2044 / 1870 rows, got {len(train)} / {len(val)} / {len(test)}"
assert train["appt_date"].max() < val["appt_date"].min() and val["appt_date"].max() < test["appt_date"].min(), "The three sets must follow each other in time"
print("OK")

## Task 3: turn the scenario into costs

For one appointment, compared with not calling:

- calling a patient who would have **attended** anyway costs the call: this is the false positive cost $c_{FP}$;
- **not** calling a patient who will **not show** misses the call's expected benefit, $0.3 \times £120$,
  minus the call's cost: this is the false negative cost $c_{FN}$.

Set `C_FP`, `C_FN` and the cost-based threshold `T_STAR` $= c_{FP} / (c_{FP} + c_{FN})$.

Then write two functions, where `y` is the no-show label (0/1) and `call` says whom we phone (booleans):

- `policy_cost(y, call)`: the expected total cost in pounds. Each call costs £8. A no-show who was called
  still costs $0.7 \times £120$ in expectation; a no-show who was not called costs £120. Attenders who were
  not called cost nothing.
- `saving_per_1000(y, call)`: how much cheaper the policy is than calling nobody, per 1,000 appointments:
  $(\text{cost(nobody)} - \text{cost(policy)}) / n \times 1000$.

In [ ]:
SLOT_VALUE, CALL_COST, CALL_EFFECT = 120, 8, 0.3

# TODO
C_FP = None
C_FN = None
T_STAR = None

def policy_cost(y, call):
    pass

def saving_per_1000(y, call):
    pass

In [ ]:
# Check
assert C_FP == 8 and abs(C_FN - 28) < 1e-9, f"Expected C_FP = 8 and C_FN = 0.3 * 120 - 8 = 28, got {C_FP} and {C_FN}"
assert abs(T_STAR - 8 / 36) < 1e-9, f"T_STAR should be 8 / 36, got {T_STAR}"
_y = np.array([1, 1, 0, 0, 1])
_c = np.array([True, False, True, False, False])
assert abs(policy_cost(_y, _c) - (2 * 8 + 0.7 * 120 + 2 * 120)) < 1e-9, f"For this small example expected {2 * 8 + 0.7 * 120 + 2 * 120}, got {policy_cost(_y, _c)}"
assert abs(policy_cost(_y, np.zeros(5, dtype=bool)) - 360) < 1e-9, "Calling nobody costs 120 per no-show"
assert abs(saving_per_1000(_y, np.zeros(5, dtype=bool))) < 1e-9, "Calling nobody saves nothing compared with itself"
assert abs(saving_per_1000(_y, _c) - (360 - policy_cost(_y, _c)) / 5 * 1000) < 1e-9, "saving_per_1000 is (cost of nobody - cost of policy) / n * 1000"
assert abs(saving_per_1000(np.array([1, 0]), np.array([True, True])) - (120 - (16 + 84)) / 2 * 1000) < 1e-9, "Check saving_per_1000 when everyone is called"
print("OK")

## Task 4: baselines, then a simple model

Compute on the **validation** set:

1. `save_everyone`: the saving per 1,000 appointments of calling everyone (calling nobody saves 0 by definition);
2. `save_rule`: the saving of the receptionists' heuristic, *call patients who have missed an appointment
   before (`prior_no_shows >= 1`) and booked more than 14 days ahead (`lead_days > 14`)*;
3. a simple model: a pipeline with a `ColumnTransformer` (scale numeric features, one-hot encode
   categorical ones, with `handle_unknown="ignore"`) and `LogisticRegression(max_iter=1000)`, fitted on
   `train[FEATURES]`. Store it in `model`, the validation probabilities in `p_val`, and the validation
   ROC AUC in `auc_val`.

In [ ]:
# TODO
save_everyone = None
save_rule = None
model = None
p_val = None
auc_val = None

In [ ]:
# Check
_yv = val["no_show"].to_numpy()
assert abs(save_everyone - saving_per_1000(_yv, np.ones(len(_yv), dtype=bool))) < 1e-6, "save_everyone is wrong"
assert save_everyone < 0, "Calling everyone should cost more than calling nobody here: check your policy_cost"
_rule = ((val["prior_no_shows"] >= 1) & (val["lead_days"] > 14)).to_numpy()
assert abs(save_rule - saving_per_1000(_yv, _rule)) < 1e-6, "save_rule should use prior_no_shows >= 1 and lead_days > 14 on the validation set"
assert p_val is not None and len(p_val) == len(val), "p_val needs one probability per validation row"
assert np.allclose(p_val, model.predict_proba(val[FEATURES])[:, 1]), "Use the model’s no-show probabilities"
assert np.isclose(auc_val, roc_auc_score(_yv, p_val)), "Use validation ROC AUC"
print("OK")

**4b. Your answer.** In one or two sentences: which baseline is the real competitor for the model, and why
is calling everyone *worse* than calling nobody here?

*Your answer:* ...

## Task 5: choose the threshold, then test once

**Success criterion, agreed before looking at the test set:** *on the test months, the model's saving per
1,000 appointments is positive and at least 1.5 times the heuristic's.*

1. Scan thresholds `np.round(np.linspace(0.02, 0.8, 79), 2)` on the **validation** set, compute the saving
   of `p_val >= t` for each, and store the best one in `t_best` (the highest saving).
2. Plot saving against threshold, and mark `T_STAR` and `t_best` with vertical lines.
3. Only now, compute on the **test** set: `save_rule_test` (the heuristic) and `save_model_test` (the model
   at `t_best`), and set `success` to whether the criterion is met.

In [ ]:
# TODO
thresholds = np.round(np.linspace(0.02, 0.8, 79), 2)
t_best = None
save_rule_test = None
save_model_test = None
success = None

In [ ]:
# Check
_yv = val["no_show"].to_numpy()
_s = [saving_per_1000(_yv, p_val >= t) for t in thresholds]
assert t_best is not None and abs(saving_per_1000(_yv, p_val >= t_best) - max(_s)) < 1e-6, "t_best should give the highest validation saving among the thresholds"
assert 0.1 <= t_best <= 0.4, f"t_best should be near T_STAR, got {t_best}"
_yt = test["no_show"].to_numpy()
_rt = ((test["prior_no_shows"] >= 1) & (test["lead_days"] > 14)).to_numpy()
assert abs(save_rule_test - saving_per_1000(_yt, _rt)) < 1e-6, "save_rule_test is the heuristic's saving on the test set"
_pt = model.predict_proba(test[FEATURES])[:, 1]
assert abs(save_model_test - saving_per_1000(_yt, _pt >= t_best)) < 1e-6, "save_model_test is the model's saving at t_best on the test set"
assert bool(success) == bool(save_model_test > 0 and save_model_test >= 1.5 * save_rule_test), "The model must save money and meet the heuristic comparison"
print("OK")

## Task 6: risks and requirements

Build `by_band`: a DataFrame with one row per `deprivation_band` on the **test** set and three columns,
`no_show_rate`, `call_rate` (the share of appointments the model at `t_best` would call), and `recall`
(the share of no-shows that would be called).

In [ ]:
# TODO
by_band = None

In [ ]:
# Check
assert isinstance(by_band, pd.DataFrame) and len(by_band) == 5, "by_band needs one row per deprivation band (5 rows)"
assert {"no_show_rate", "call_rate", "recall"} <= set(by_band.columns), "by_band needs the columns no_show_rate, call_rate and recall"
_t = test.assign(call=model.predict_proba(test[FEATURES])[:, 1] >= t_best)
for _b in range(1, 6):
    _g = _t[_t["deprivation_band"] == _b]
    assert abs(by_band.loc[_b, "call_rate"] - _g["call"].mean()) < 1e-9, f"call_rate for band {_b} is wrong"
    assert abs(by_band.loc[_b, "recall"] - _g.loc[_g["no_show"] == 1, "call"].mean()) < 1e-9, f"recall for band {_b} is wrong"
    assert abs(by_band.loc[_b, "no_show_rate"] - _g["no_show"].mean()) < 1e-9, f"no_show_rate for band {_b} is wrong"
print("OK")

**6b. Your answer.** Answer each in one to three sentences.

1. Look at `by_band`. A reminder call is a *benefit* to the patient. Is the difference in call rates
   between bands a fairness problem here? Would your answer change if the clinic used the same scores to
   **double-book** the slots of likely no-shows?
2. Batch or online prediction? How often should predictions be made, and why?
3. Once calls start, how will the labels in next year's data differ from this year's? What would you
   do so that you can still retrain and evaluate fairly? (Lesson 4: feedback loops.)
4. Name two failure modes and the fallback for each.
5. Which personal data does the model use? Is each column needed?

*Your answers:* ...

## Task 7: the one-page design doc

Fill in the design doc below with **your** numbers from this notebook. Aim for one page: short sentences
and bullet points, no code. Someone who has not seen this notebook should be able to review it.

### ML design doc: reducing appointment no-shows

**1. Problem and objective.** ...

**2. Decision and users.** ...

**3. ML task.** Target, unit of prediction, prediction time, label definition, label delay (median and p95), proxy gaps. ...

**4. Data.** Sources, volumes, features used and excluded (and why), personal data. ...

**5. Metrics and success criteria.** Offline metric, online metrics and guardrails, the success threshold. ...

**6. Baselines.** Calling nobody, calling everyone, the heuristic, the model: validation and test numbers. ...

**7. System sketch.** Batch or online, cadence, features and their freshness, logging, holdout, monitoring. ...

**8. Risks and requirements.** Fairness, privacy, explainability for receptionists, failure modes and fallbacks. ...

**9. Rollout plan.** ...

**10. Open questions.** ...

## Stretch tasks (optional)

- **Capacity.** Receptionists can make at most 15 calls per clinic day. Rank each day's appointments by
  `p_test` and call the top 15. How does the saving compare with the threshold policy? Which offline
  metric matches this decision?
- **Calibration.** Compare the mean of `p_val` with the validation no-show rate, and plot a reliability
  curve (`sklearn.calibration.calibration_curve`). Does that explain any gap between `T_STAR` and `t_best`?
- **Sensitivity.** The 30% call effect comes from a small pilot. Recompute `t_best` and the test saving for
  effects of 20% and 40%. How robust is the decision to launch?